# Phase 2 — Baseline Evaluation

This notebook is an interactive interface to the tested Phase 2 runner. It evaluates all 15 ground-truth questions against the existing Phase 1 dense Pinecone namespace and displays the recorded Recall@5 summary.

Prerequisites: populate `.env`, keep Ollama running, and index the Phase 1 namespace first. The runner is read-only with respect to that namespace.

## Setup

Use the repository `.venv` kernel so the notebook and command-line runner share dependencies.

In [ ]:
from __future__ import annotations

import json
import shlex
import subprocess
import sys
from pathlib import Path

import yaml
from IPython.display import JSON, display

PROJECT_ROOT = next(
    (
        candidate.resolve()
        for candidate in (Path.cwd(), Path.cwd().parent)
        if (candidate / "pyproject.toml").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from the repository root or notebooks directory.")


def run_command(command: list[object]) -> None:
    normalized = [str(part) for part in command]
    print(shlex.join(normalized))
    subprocess.run(normalized, cwd=PROJECT_ROOT, check=True)


def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def read_yaml(path: Path) -> dict:
    return yaml.safe_load(path.read_text(encoding="utf-8")) or {}


print(f"Project root: {PROJECT_ROOT}")
print(f"Kernel Python: {sys.executable}")


## Parameters

External execution is opt-in. Existing artifacts can still be inspected with the switch left off.

In [ ]:
EXPERIMENT_ID = "E001_dense_baseline"
EXPERIMENT_NAME = "Phase 2 dense baseline"
QUESTIONS_PATH = PROJECT_ROOT / "evaluation" / "questions.json"
OUTPUT_ROOT = PROJECT_ROOT / "evaluation" / "results"
RUN_EXPERIMENT = False  # Set to True, then run the next cell.

print(f"Questions: {QUESTIONS_PATH}")
print(f"Results: {OUTPUT_ROOT / EXPERIMENT_ID}")

## Run the controlled evaluation

In [ ]:
COMMAND = [
    sys.executable,
    PROJECT_ROOT / "evaluation" / "run_baseline.py",
    "--experiment-id", EXPERIMENT_ID,
    "--experiment-name", EXPERIMENT_NAME,
    "--questions", QUESTIONS_PATH,
    "--output-root", OUTPUT_ROOT,
]

if RUN_EXPERIMENT:
    run_command(COMMAND)
else:
    print("Dry run. Set RUN_EXPERIMENT = True to execute:")
    print(shlex.join(str(part) for part in COMMAND))

## Inspect the result

In [ ]:
RESULTS_PATH = OUTPUT_ROOT / EXPERIMENT_ID / "results.json"

if RESULTS_PATH.exists():
    results = read_json(RESULTS_PATH)
    display(JSON({
        "experiment_id": results["experiment_id"],
        "summary": results["summary"],
        "category_summary": results["category_summary"],
    }))
else:
    print(f"No result artifact yet: {RESULTS_PATH}")